# Imabari Q&A v4 を Hugging Face にアップロード

全レコードを保持し、各行に一意なUUIDv4の `qa_id` を付けます。`id + chunk_index` および内容の重複は許可します。

In [1]:
from getpass import getpass
import json
import os
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'main_upload_dataset.py').is_file():
    raise RuntimeError('リポジトリのルートディレクトリで実行してください。')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from main_create_imabari_qa import new_qa_id
from main_upload_dataset import DEFAULT_EXCLUDE_UPLOAD_KEYS, main as upload_dataset

SOURCE_FILE = PROJECT_ROOT / 'test_output/imabari_qa_v4_merged/all.jsonl'
UPLOAD_DIR = PROJECT_ROOT / 'test_output/imabari_qa_v4_merged/upload_ready'
REPO_ID = 'ikedachin/imabari_wiki_qa_v4'
PRIVATE = True
COMMIT_MESSAGE = 'Upload Imabari Q&A dataset'
VALIDATION_RATIO = 0.1
SPLIT_SEED = 42

print(f'Source: {SOURCE_FILE}')
print(f'Repository: {REPO_ID}')

Source: /Users/ikedashinji/Desktop/holon_workspace/imabarize/test_output/imabari_qa_v4_merged/all.jsonl
Repository: ikedachin/imabari_wiki_qa_v4


## 1. 全レコードを保持したアップロード用JSONLの作成

In [2]:
records = []
with SOURCE_FILE.open('r', encoding='utf-8') as f:
    for line_no, line in enumerate(f, 1):
        if not line.strip():
            continue
        record = json.loads(line)
        if not isinstance(record, dict):
            raise ValueError(f'Line {line_no}: JSON objectではありません。')
        if record.get('id') in (None, ''):
            raise ValueError(f'Line {line_no}: idがありません。')
        records.append(record)

used_qa_ids = set()
prepared_records = []
assigned_or_repaired = 0

for record in records:
    qa_id = record.get('qa_id')
    if not qa_id or qa_id in used_qa_ids:
        qa_id = new_qa_id()
        while qa_id in used_qa_ids:
            qa_id = new_qa_id()
        assigned_or_repaired += 1
    used_qa_ids.add(qa_id)

    upload_record = dict(record)
    upload_record.pop('item_id', None)
    upload_record['qa_id'] = qa_id
    prepared_records.append(upload_record)

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
upload_file = UPLOAD_DIR / 'all.jsonl'
with upload_file.open('w', encoding='utf-8') as f:
    for record in prepared_records:
        json.dump(record, f, ensure_ascii=False)
        f.write('\n')

assert len(prepared_records) == len(records)
assert len({record['qa_id'] for record in prepared_records}) == len(prepared_records)
assert all('item_id' not in record for record in prepared_records)

print(f'Prepared: {upload_file}')
print(f'Records retained: {len(prepared_records):,} / {len(records):,}')
print(f'Unique qa_id: {len(used_qa_ids):,}')
print(f'qa_id assigned or repaired: {assigned_or_repaired:,}')

Prepared: /Users/ikedashinji/Desktop/holon_workspace/imabarize/test_output/imabari_qa_v4_merged/upload_ready/all.jsonl
Records retained: 8,319 / 8,319
Unique qa_id: 8,319
qa_id assigned or repaired: 8,319


## 2. 9:1分割のDry run

全レコードを固定seedでシャッフルし、`train.jsonl` 90%・`validation.jsonl` 10%に分割します。

In [3]:
upload_dataset(
    repo_id=REPO_ID,
    hf_token=None,
    dataset_path=str(UPLOAD_DIR),
    settings_path=None,
    private=PRIVATE,
    include_splits=False,
    dry_run=True,
    commit_message=COMMIT_MESSAGE,
    dataset_kind='qa',
    split_on_upload=True,
    validation_ratio=VALIDATION_RATIO,
    split_seed=SPLIT_SEED,
    exclude_upload_keys=DEFAULT_EXCLUDE_UPLOAD_KEYS,
)

💡 [INFO] 2026-08-28 14:22:10 dataset_dir=/Users/ikedashinji/Desktop/holon_workspace/imabarize/test_output/imabari_qa_v4_merged/upload_ready
💡 [INFO] 2026-08-28 14:22:10 upload_file=train.jsonl records=7488
💡 [INFO] 2026-08-28 14:22:10 upload_file=validation.jsonl records=831
✅ [SUCCESS] 2026-08-28 14:22:10 Dry run completed. No files were uploaded.


## 3. Hugging Faceへアップロード

次のセルを実行すると、シャッフル済みの `train.jsonl` と `validation.jsonl` がHugging Faceへアップロードされます。

In [4]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()
hf_token = (os.environ.get('HF_TOKEN') or '').strip()
if not hf_token:
    hf_token = getpass('Hugging Face Token: ')
    if not hf_token:
        raise RuntimeError('Hugging Face Tokenが指定されていません。')
print('Hugging Face Tokenを読み込みました。')

Hugging Face Tokenを読み込みました。


In [5]:
REPO_ID

'ikedachin/imabari_wiki_qa_v4'

In [6]:


upload_dataset(
    repo_id=REPO_ID,
    hf_token=hf_token,
    dataset_path=str(UPLOAD_DIR),
    settings_path=None,
    private=PRIVATE,
    include_splits=False,
    dry_run=False,
    commit_message=COMMIT_MESSAGE,
    dataset_kind='qa',
    split_on_upload=True,
    validation_ratio=VALIDATION_RATIO,
    split_seed=SPLIT_SEED,
    exclude_upload_keys=DEFAULT_EXCLUDE_UPLOAD_KEYS,
)
hf_token = None

💡 [INFO] 2026-08-28 14:22:10 dataset_dir=/Users/ikedashinji/Desktop/holon_workspace/imabarize/test_output/imabari_qa_v4_merged/upload_ready
💡 [INFO] 2026-08-28 14:22:10 upload_file=train.jsonl records=7488
💡 [INFO] 2026-08-28 14:22:10 upload_file=validation.jsonl records=831
💡 [INFO] 2026-08-28 14:22:11 Uploading /var/folders/kf/r8kkkdns4_75ynr7vrp1jnwc0000gn/T/dataset_upload_j589je2d/train.jsonl -> ikedachin/imabari_wiki_qa_v4/train.jsonl


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

💡 [INFO] 2026-08-28 14:22:16 Uploading /var/folders/kf/r8kkkdns4_75ynr7vrp1jnwc0000gn/T/dataset_upload_j589je2d/validation.jsonl -> ikedachin/imabari_wiki_qa_v4/validation.jsonl
✅ [SUCCESS] 2026-08-28 14:22:18 Uploaded dataset: https://huggingface.co/datasets/ikedachin/imabari_wiki_qa_v4
